MovieLens Dataset Exploration

In [9]:
import pandas as pd
from pathlib import Path

Locate the dataset

In [10]:
DATA_PATH = Path("../../data/")

movies_path = DATA_PATH / "movies.csv"
ratings_path = DATA_PATH / "ratings.csv"

print("Movies file exists:", movies_path.exists())
print("Ratings file exists:", ratings_path.exists())

Movies file exists: True
Ratings file exists: True


Load the datasets

In [11]:
# Load movie information
movies_df = pd.read_csv(movies_path)

# Load user ratings
ratings_df = pd.read_csv(ratings_path)

print("Movies Dataset:")
display(movies_df.head())

print("Ratings Dataset:")
display(ratings_df.head())

Movies Dataset:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Ratings Dataset:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Dataset dimensions

In [12]:
print("Movies shape:", movies_df.shape)
print("Ratings shape:", ratings_df.shape)

Movies shape: (9742, 3)
Ratings shape: (100836, 4)


Column information

In [13]:
print("Movies information:")
movies_df.info()

print("\nRatings information:")
ratings_df.info()

Movies information:
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB

Ratings information:
<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


Statistical summary

In [14]:
print("Movies summary:")
display(movies_df.describe(include="all"))

print("Ratings summary:")
display(ratings_df.describe())

Movies summary:


,movieId,title,genres
count,9742.000000,9742,9742
unique,NaN,9737,951
top,NaN,Emma (1996),Drama
freq,NaN,2,1053
mean,42200.353623,NaN,NaN
std,52160.494854,NaN,NaN
min,1.000000,NaN,NaN
25%,3248.250000,NaN,NaN
50%,7300.000000,NaN,NaN
75%,76232.000000,NaN,NaN


Ratings summary:


,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


Count unique users and movies

In [15]:
total_users = ratings_df["userId"].nunique()
total_movies = movies_df["movieId"].nunique()
total_ratings = len(ratings_df)

print("Total users:", total_users)
print("Total movies:", total_movies)
print("Total ratings:", total_ratings)

Total users: 610
Total movies: 9742
Total ratings: 100836


Understand rating distribution

In [16]:
print(ratings_df["rating"].value_counts().sort_index())

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


Calculate movie statistics

In [18]:
movie_stats=ratings_df.groupby("movieId").agg(average_rating=("rating","mean"),rating_count=("rating","count")).reset_index()
movie_stats.head()

,movieId,average_rating,rating_count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49


- C = Average rating across all movies
- m = Minimum rating count (90th percentile)

In [19]:
C=ratings_df["rating"].mean()
m=movie_stats["rating_count"].quantile(0.90)

print("Overall average rating:", C)
print("Minimum rating count:", m)

Overall average rating: 3.501556983616962
Minimum rating count: 27.0


Calculate weighted rating

In [20]:
# Calculate weighted rating for every movie
movie_stats["weighted_rating"] = (
    (movie_stats["rating_count"] / (movie_stats["rating_count"] + m))
    * movie_stats["average_rating"]
    +
    (m / (movie_stats["rating_count"] + m))
    * C
)

movie_stats.head()

,movieId,average_rating,rating_count,weighted_rating
0,1,3.920930,215,3.874141
1,2,3.431818,110,3.445562
2,3,3.259615,52,3.342304
3,4,2.357143,7,3.265942
4,5,3.071429,49,3.224237


Merge movie information

In [21]:
popular_movies=movies_df.merge(movie_stats,on="movieId",how="inner")
popular_movies.head()

,movieId,title,genres,average_rating,rating_count,weighted_rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215,3.874141
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110,3.445562
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52,3.342304
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7,3.265942
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49,3.224237


Generate recommendations

In [22]:
# Recommend movies with sufficient ratings

popular_movies = popular_movies[
    popular_movies["rating_count"] >= m
]

# Sort by weighted rating
popular_movies=popular_movies.sort_values(by="weighted_rating",ascending=False)

# Display Top 10 recommendations

top_10_movies = popular_movies[
    ["title", "genres", "average_rating", "rating_count", "weighted_rating"]
].head(10)

top_10_movies

,title,genres,average_rating,rating_count,weighted_rating
277,"Shawshank Redemption, The (1994)",Crime|Drama,4.429022,317,4.356227
659,"Godfather, The (1972)",Crime|Drama,4.289062,192,4.191973
2224,Fight Club (1999),Action|Crime|Drama|Thriller,4.272936,218,4.187927
224,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.231076,251,4.160223
46,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.237745,204,4.151697
461,Schindler's List (1993),Drama|War,4.225000,220,4.145919
257,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.197068,307,4.140844
897,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.215640,211,4.134630
1938,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.192446,278,4.131285
921,"Godfather: Part II, The (1974)",Crime|Drama,4.259690,129,4.128475
